In [5]:
from docx import Document
import re
import tkinter as tk
from tkinter import filedialog
import os

# Define the abbreviation replacements
replacements = {
    'NCCB': 'pre-alcohol non-conflict',
    'CCB': 'pre-alcohol conflict',
    'AA': 'alcohol',
    'NCPA': 'distal-post-alcohol non-conflict',
    'CPA': 'distal-post-alcohol conflict',
    'PNC': 'proximal-post-alcohol non-conflict',
    'PC': 'proximal-post-alcohol conflict'
}

# Pattern to match figure legends like (a), (b), ..., (z)
legend_pattern = re.compile(r"\([a-z]\)")

def bold_legends_in_paragraph(para):
    """Rebuilds a paragraph to bold figure legends like (a), (b) while preserving other formatting."""
    runs = para.runs
    new_text = para.text  # Full paragraph text
    
    # Find all matches in the text
    matches = legend_pattern.findall(new_text)
    
    if matches:
        para.clear()  # Remove all runs
        
        last_index = 0
        for match in matches:
            start_idx = new_text.find(match, last_index)
            if start_idx > last_index:
                para.add_run(new_text[last_index:start_idx])  # Add normal text
            
            bold_run = para.add_run(match)  # Add the matched text as a new run
            bold_run.bold = True  # Make it bold
            
            last_index = start_idx + len(match)

        # Add any remaining text after the last match
        if last_index < len(new_text):
            para.add_run(new_text[last_index:])

def replace_text_in_docx(input_file, output_file):
    """Replaces abbreviations and bolds figure legends in a .docx document."""
    doc = Document(input_file)
    
    # Process text in paragraphs
    for para in doc.paragraphs:
        for key, value in replacements.items():
            if key in para.text:
                para.text = para.text.replace(key, value)

        # Apply bold formatting to figure legends
        if legend_pattern.search(para.text):
            bold_legends_in_paragraph(para)

    # Process text in tables (if any)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for key, value in replacements.items():
                    if key in cell.text:
                        cell.text = cell.text.replace(key, value)

                # Apply bold formatting to figure legends in table cells
                for para in cell.paragraphs:
                    if legend_pattern.search(para.text):
                        bold_legends_in_paragraph(para)

    # Save the modified document
    doc.save(output_file)
    print(f"Updated document saved as: {output_file}")

# Example usage with file dialog
root = tk.Tk()
root.withdraw()  # Hide the main tkinter window

# User selects the input file
input_docx = filedialog.askopenfilename(title="Select Input .docx File", filetypes=[("Word Documents", "*.docx")])
print(f"Selected input file: {input_docx}")

# Save output in the same directory as input file
output_docx = os.path.join(os.path.dirname(input_docx), "output.docx")

# Apply text replacement and bold formatting
replace_text_in_docx(input_docx, output_docx)


Selected input file: /Users/atanugiri/Desktop/Fig_legend.docx
Updated document saved as: /Users/atanugiri/Desktop/output.docx
